# Wopke MAS (Label-First, Inspectable)

This notebook uses an explicit five-stage flow so every intermediate artifact is easy to inspect:

1. **Value extraction** — `value_identifier` scans the full paper for all field-value pairs (high recall).
2. **Field XML labeling** — `labeller` wraps matched values in the paper with `<FieldName>` XML tags.
3. **Record grouping** — `record_grouper` reads the labeled paper and produces **schema-conformant
   candidate records** (same fields as the final schema + `record_confidence`), one per treatment
   combination. Explicitly handles multi-record designs (e.g. 4×4 factorial → 16 records).
4. **Record XML labeling** — `record_labeller` reads the candidate records and wraps each
   corresponding table row in the paper with `<Record_N>` tags.
5. **Final structured extraction** — `record_extractor` reads the doubly-labeled paper + candidate record
   scaffold and produces the final schema-conformant output.


In [1]:
# Setup
import os
import re
import sys
import json
from typing import Any, Dict, List

sys.path.insert(0, '..')

import pandas as pd

from src.context import create_context
from src.tools.context_tools import register_context, clear_registry
from src.players import PLAYER_CONFIGS, create_player_from_config
from src.standards import METADATA_STANDARDS
from src.experimentutils import (
    load_ground_truth,
    build_study_paper_mapping,
    read_paper_text,
    highlight_numbers_and_tables,
    save_extraction_results_with_timestamp,
)
from pydantic import create_model
from src.direct_llm_call.schemas import create_extraction_schema, create_record_schema

print('Imports OK')

Imports OK


In [2]:
def _extract_field_value_pairs(raw_output: Any) -> List[List[str]]:
    """Parse value_identifier output into [[field, value], ...]."""
    if isinstance(raw_output, list):
        data = raw_output
    else:
        s = str(raw_output).strip()
        s = re.sub(r'^```json\s*', '', s)
        s = re.sub(r'^```\s*', '', s)
        s = re.sub(r'\s*```$', '', s).strip()
        try:
            data = json.loads(s)
        except Exception:
            m = re.search(r'(\[[\s\S]*\]|\{[\s\S]*\})', s)
            if not m:
                return []
            data = json.loads(m.group(1))

    # Expected main format: [[field, value], ...]
    if isinstance(data, list):
        pairs = []
        for item in data:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                f = str(item[0]).strip()
                v = str(item[1]).strip()
                if f and v:
                    pairs.append([f, v])
        return pairs

    # Fallback format: {"field_value_candidates": [{"field":..., "value":...}, ...]}
    if isinstance(data, dict) and isinstance(data.get('field_value_candidates'), list):
        pairs = []
        for item in data['field_value_candidates']:
            if not isinstance(item, dict):
                continue
            f = str(item.get('field', '')).strip()
            v = str(item.get('value', '')).strip()
            if f and v:
                pairs.append([f, v])
        return pairs

    return []


def _pairs_to_candidates(pairs: List[List[str]]) -> List[Dict[str, Any]]:
    """Convert pairs into candidate objects used downstream."""
    out: List[Dict[str, Any]] = []
    for f, v in pairs:
        out.append({
            'field': f,
            'value': v,
            'confidence': 0.7,
            'evidence': 'value_identifier',
        })
    return out


def _dedupe_pairs(pairs: List[List[str]]) -> List[List[str]]:
    seen = set()
    uniq = []
    for f, v in pairs:
        k = (f.strip().lower(), re.sub(r'\s+', ' ', v.strip().lower()))
        if k in seen:
            continue
        seen.add(k)
        uniq.append([f, v])
    return uniq


def _to_field_value_pairs(candidates: List[Dict[str, Any]]) -> List[List[str]]:
    pairs = []
    for item in candidates:
        field = str(item.get('field', '')).strip()
        value = str(item.get('value', '')).strip()
        if field and value:
            pairs.append([field, value])
    return pairs


In [3]:
# Select study and build context from Wopke dataset
study_id = 3
gt_df = load_ground_truth()
mapping = build_study_paper_mapping(gt_df)

if study_id not in mapping:
    raise ValueError(f'Study# {study_id} not found in mapping')

paper_path = mapping[study_id]
paper_name = os.path.basename(paper_path)
print(f'Study# {study_id} -> {paper_name}')

data_context = create_context(source=paper_path, name=f'wopke_study_{study_id}')
context_key = f'wopke_ctx_{study_id}'
clear_registry()
register_context(context_key, data_context)

resource_name = data_context.resources[0]
print(f'Context resource: {resource_name}')

Study# 3 -> Bulson 1997 Effects of plant density on intercropped wheat and field beans in an organic farming system.md
Context resource: Bulson 1997 Effects of plant density on intercropped wheat and field beans in an organic farming system


In [4]:
# Prepare schema and prompts
standard = METADATA_STANDARDS['wopke_100']
raw_text = read_paper_text(paper_path)
highlighted = highlight_numbers_and_tables(raw_text)
print(f'Document length: {len(raw_text):,}')
print(f'Highlighted length: {len(highlighted):,}')

# ── Schema-based candidate record models (built once standard is loaded) ────
from pydantic import BaseModel, Field as PydanticField

_WopkeRecordBase = create_record_schema(standard, 'WopkeRecordBase')

WopkeRecordCandidate = create_model(
    'WopkeRecordCandidate',
    __base__=_WopkeRecordBase,
    record_confidence=(float, PydanticField(
        0.5, ge=0.0, le=1.0,
        description="Confidence 0–1: 1.0=explicit table cell, 0.8=inferred, 0.5=ambiguous")),
    treatment_description=(str, PydanticField(
        '',
        description="Short description of the treatment combination for this record, "
                    "e.g. 'wheat 50%RD × beans 75%RD'")),
)


class WopkeCandidateOutput(BaseModel):
    experimental_design: str = PydanticField(
        ..., description="Experimental design, e.g. '4-level wheat × 4-level bean factorial'")
    total_records_expected: int = PydanticField(
        ..., ge=1,
        description="Total distinct records implied by the design (e.g. 4×4=16)")
    design_reasoning: str = PydanticField(
        ...,
        description="Reasoning for the record count, e.g. '4 wheat densities × 4 bean densities = 16'")
    yield_records: List[WopkeRecordCandidate] = PydanticField(
        ...,
        description="ALL candidate records — one per treatment combination. "
                    "Must contain total_records_expected entries.")


# ── Final extraction schema (used in Step 5) ────────────────────────────────
output_schema = create_extraction_schema(
    standard=standard,
    record_class_name='WopkeRecord',
    output_class_name='WopkeOutput',
    records_key='yield_records',
)

final_llm_base = PLAYER_CONFIGS['record_extractor']['role_prompt']
final_structuring_prompt = f"""
{final_llm_base}

META-ANALYTIC SCHEMA:
{standard}
"""

print('Schema models built OK')


Document length: 61,496
Highlighted length: 79,886
Schema models built OK


In [5]:
# Step 1: value extraction using value_identifier player (high recall)
workspace = {
    'meta_analytic_schema': standard,
}

value_identifier = create_player_from_config(
    PLAYER_CONFIGS['value_identifier'],
    name='value_identifier',
)

value_identifier_task = f"""
Extract field-value pairs for WOPKE_100 with MAXIMUM COVERAGE.

Important:
- Return ALL field-value pairs you can find across full paper content.
- Focus especially on row-level table values for:
  Crop species 1/2, unified yield sc 1/2, unified yield ic 1/2,
  Density ic/sc 1/2, Year of data, Yield unit, Data source.
- Keep scanning until complete; do not stop after first few values.

Schema reference:
{standard}
"""

value_out = value_identifier.execute_task(
    task=value_identifier_task,
    context_key=context_key,
    context_info=data_context.to_dict(),
    workspace=workspace,
    inputs={},
    target_resources=[resource_name],
)

raw_candidates_output = value_out.get('analysis', '')
field_value_pairs = _extract_field_value_pairs(raw_candidates_output)
field_value_pairs = _dedupe_pairs(field_value_pairs)
field_value_candidates = _pairs_to_candidates(field_value_pairs)

workspace['field_value_candidates'] = field_value_candidates
workspace['field_value_pairs'] = field_value_pairs

print(f"Pairs extracted: {len(field_value_pairs)}")
print(f"Tag pairs      : {len(field_value_pairs)}")

cand_df = pd.DataFrame(field_value_candidates)
if not cand_df.empty:
    display(cand_df['field'].value_counts().head(20))

display(cand_df.head(30))

Pairs extracted: 27
Tag pairs      : 27


field
Data source              15
Experimental design       1
Sowing date 1             1
Sowing date 2             1
Year of data              1
Harvest date 1            1
Harvest date 2            1
Crop species 2            1
Crop species 1            1
Crop type 1               1
Crop type 2               1
Intercropping pattern     1
Yield unit                1
Name: count, dtype: int64

,field,value,confidence,evidence
0,Year of data,1987–88,0.7,value_identifier
1,Experimental design,bivariate factorial design,0.7,value_identifier
2,Sowing date 1,field beans sown 4 November 1987,0.7,value_identifier
3,Sowing date 2,wheat sown 5 November 1987,0.7,value_identifier
4,Harvest date 1,3 October 1988,0.7,value_identifier
5,Harvest date 2,3 October 1988,0.7,value_identifier
6,Crop species 1,wheat,0.7,value_identifier
7,Crop species 2,field beans,0.7,value_identifier
8,Crop type 1,cereal,0.7,value_identifier
9,Crop type 2,legume,0.7,value_identifier


In [6]:
# Step 2: full-document XML labeling via labeller player config
labeller = create_player_from_config(PLAYER_CONFIGS['labeller'], name='labeller_tool')

label_out = labeller.execute_task(
    task='Label the full paper with XML tags using field_value_pairs.',
    context_key=context_key,
    context_info=data_context.to_dict(),
    workspace=workspace,
    inputs={'field_value_pairs': 'field_value_pairs'},
    target_resources=[resource_name],
)

labeled_text = label_out['analysis']
workspace['labeled_text'] = labeled_text

print(f'Labeled text length: {len(labeled_text):,}')
print('Tagged snippet preview:')
print(labeled_text[:2000])

Labeled text length: 69,915
Tagged snippet preview:
# Effects of plant density on intercropped <Crop species 1>wheat</Crop species 1> and <Crop species 2>field beans</Crop species 2> in an organic farming system

H. A. J. BULSON*, R. W. SNAYDON  C. E. STOPES*

Agricultural Botany Department, University of Reading, Reading RG6 2AS, UK (Revised MS received 8 January 1996)

# SUMMARY

In field trials in 1987}88 near Pangbourne, England, <Crop species 1>wheat</Crop species 1> (Triticum aestivum) and <Crop species 2>field beans</Crop species 2> (Vicia faba) were grown in an organic farming system as sole crops and additive intercrops. The sole crops were grown at 25, 50, 75, 100 and $1 5 0 \%$ of the recommended density (RD) for conventionally grown crops. The intercrops consisted of all density combinations of <Crop species 1>wheat</Crop species 1> and beans from 25 to $100 \%$ RD in a factorial experiment. The grain yield of sole cropped <Crop species 1>wheat</Crop species 1> and beans

In [7]:
# Step 3: record grouping — produce schema-conformant candidate records with confidence
record_grouper = create_player_from_config(
    PLAYER_CONFIGS['record_grouper'],
    name='record_grouper',
)

record_grouper_prompt = f"""
{PLAYER_CONFIGS['record_grouper']['role_prompt']}

META-ANALYTIC SCHEMA (use EXACTLY these field names in every record):
{standard}

CANDIDATE FIELD-VALUE PAIRS (from Step 1, for reference):
{json.dumps(field_value_candidates, ensure_ascii=False, indent=2)}

FULL LABELED PAPER CONTENT:
---
{labeled_text}
---

TASK:
1. Identify the experimental design (e.g. 4 wheat densities × 4 bean densities = 16 records).
2. Fill `total_records_expected` with that count.
3. Enumerate EVERY treatment combination as a separate record in `yield_records`.
   - Shared fields (Year, Crop species, Yield unit, etc.) are identical in every record.
   - Treatment-specific fields (Density ic 1/2, unified yield ic 1/2, etc.) differ per row.
4. Assign `record_confidence` and `treatment_description` to each record.
5. Use null for fields you cannot determine — do NOT leave out records.

CRITICAL: `yield_records` MUST contain ALL {'{total_records_expected}'} records.
Do NOT collapse table rows — each row = one record.
"""

structured_record_grouper = record_grouper.llm.with_structured_output(WopkeCandidateOutput)
grouped_records: WopkeCandidateOutput = structured_record_grouper.invoke(record_grouper_prompt)

workspace['grouped_records'] = grouped_records.model_dump()

print(f"Experimental design : {grouped_records.experimental_design}")
print(f"Expected records    : {grouped_records.total_records_expected}")
print(f"Design reasoning    : {grouped_records.design_reasoning}")
print(f"Candidate records   : {len(grouped_records.yield_records)}")
print()
for rec in grouped_records.yield_records[:6]:
    yic1 = getattr(rec, 'unified yield ic 1', None) or rec.model_dump().get('unified yield ic 1')
    print(f"  [{rec.treatment_description}]  conf={rec.record_confidence:.2f}  "
          f"yield_ic1={yic1!r}")

import pandas as pd
df_candidates = pd.DataFrame([r.model_dump() for r in grouped_records.yield_records])
print(f"\nCandidate records DataFrame: {df_candidates.shape}")
df_candidates.head(20)

Experimental design : 4 wheat densities × 4 bean densities factorial
Expected records    : 16
Design reasoning    : The experiment tested all combinations of 4 wheat densities (25, 50, 75, 100% RD) and 4 bean densities (25, 50, 75, 100% RD), resulting in 4×4=16 intercropping treatment combinations.
Candidate records   : 16

  [wheat 25%RD × beans 25%RD]  conf=0.80  yield_ic1=None
  [wheat 50%RD × beans 25%RD]  conf=0.80  yield_ic1=None
  [wheat 75%RD × beans 25%RD]  conf=0.80  yield_ic1=None
  [wheat 100%RD × beans 25%RD]  conf=0.80  yield_ic1=None
  [wheat 25%RD × beans 50%RD]  conf=0.80  yield_ic1=None
  [wheat 50%RD × beans 50%RD]  conf=0.80  yield_ic1=None

Candidate records DataFrame: (16, 44)


,Year of data,Duration of experiment,Experimental design,Sowing date 1,Sowing date 2,Harvest date 1,Harvest date 2,Lat,Lon,Crop species 1,...,K total in IC,K Unit,Data source,unified yield sc 1,unified yield sc 2,unified yield ic 1,unified yield ic 2,Yield unit,record_confidence,treatment_description
0,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 25%RD × beans 25%RD
1,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 50%RD × beans 25%RD
2,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 75%RD × beans 25%RD
3,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 100%RD × beans 25%RD
4,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 25%RD × beans 50%RD
5,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 50%RD × beans 50%RD
6,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 75%RD × beans 50%RD
7,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 100%RD × beans 50%RD
8,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 25%RD × beans 75%RD
9,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1,0.8,wheat 50%RD × beans 75%RD


In [8]:
# Step 4: record labelling — deterministic, no LLM needed
# Directly call the tool with the outputs of Steps 2 and 3.
from src.tools.xml_tagging import apply_record_tags_to_content

candidate_records_for_tool = []
for i, rec in enumerate(grouped_records.yield_records, start=1):
    d = rec.model_dump()
    d.setdefault('record_index', i)
    candidate_records_for_tool.append(d)

record_labeled_text = apply_record_tags_to_content(
    labeled_text=labeled_text,
    candidate_records=candidate_records_for_tool,
)

workspace['record_labeled_text'] = record_labeled_text

print(f'Record-labeled text length: {len(record_labeled_text):,}')
print(f'Records tagged            : {len(candidate_records_for_tool)}')

first_tag = re.search(r'<Record_\d+>', record_labeled_text)
if first_tag:
    start = max(0, first_tag.start() - 100)
    print('\nFirst Record tag context:')
    print(record_labeled_text[start:start + 600])
else:
    print('\n[No <Record_N> tags found in output]')
    print(record_labeled_text[:800])

Record-labeled text length: 69,936
Records tagged            : 16

First Record tag context:
rtment, University of Reading, Reading RG6 2AS, UK (Revised MS received 8 January 1996)

# SUMMARY

<Record_1>In field trials in 1987}88 near Pangbourne, England, <Crop species 1>wheat</Crop species 1> (Triticum aestivum) and <Crop species 2>field beans</Crop species 2> (Vicia faba) were grown in an organic farming system as sole crops and additive intercrops. The sole crops were grown at 25, 50, 75, 100 and $1 5 0 \%$ of the recommended density (RD) for conventionally grown crops. The intercrops consisted of all density combinations of <Crop species 1>wheat</Crop species 1> and beans from 25 


In [9]:
# Step 5: final structured extraction via record_extractor player config
final_player = create_player_from_config(PLAYER_CONFIGS['record_extractor'], name='record_extractor')

final_prompt = f"""
{final_structuring_prompt}

━━━ STEP 3 CANDIDATE RECORDS (USE AS SCAFFOLD) ━━━
Experimental design : {grouped_records.experimental_design}
Expected records    : {grouped_records.total_records_expected}
Design reasoning    : {grouped_records.design_reasoning}

Candidate records produced by the record_grouper (JSON).
Each entry is a near-complete record — use its field values as a starting point,
then verify / correct using the Record-labeled paper below:
{json.dumps(grouped_records.model_dump()['yield_records'], ensure_ascii=False, indent=2)}

IMPORTANT:
- You MUST produce exactly {grouped_records.total_records_expected} output records
  (or fewer only if the paper clearly does not support that many).
- Do NOT merge treatment rows — each <Record_N>...</Record_N> block in the paper
  below is a distinct record.
- Shared fields (Year, Crop species, Yield unit, etc.) must be copied into every record.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RECORD-LABELED PAPER (field XML tags + <Record_N> section markers):
---
{record_labeled_text}
---
"""

structured_final_llm = final_player.llm.with_structured_output(output_schema)
result_final = structured_final_llm.invoke(final_prompt)

workspace['final_meta_analysis_records'] = result_final.model_dump()

df_final = pd.DataFrame([r.model_dump() for r in result_final.yield_records])
print(f'Final records: {len(df_final)}')
df_final.head(20)

Final records: 16


,Year of data,Duration of experiment,Experimental design,Sowing date 1,Sowing date 2,Harvest date 1,Harvest date 2,Lat,Lon,Crop species 1,...,K input IC1,K input IC2,K total in IC,K Unit,Data source,unified yield sc 1,unified yield sc 2,unified yield ic 1,unified yield ic 2,Yield unit
0,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
1,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
2,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
3,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
4,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
5,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
6,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
7,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
8,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1
9,1987–88,1 growing season,bivariate factorial design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1


In [ ]:
# Save final results and inspect intermediate artifacts
path_final = save_extraction_results_with_timestamp(
    results=result_final,
    base_name=f'{study_id}_mas_label_first_wopke100',
    records_key='yield_records',
    include_time=False,
)
print(f'Saved: {path_final}')

print('\nWorkspace keys:')
for k in workspace.keys():
    print(' -', k)

print('\nCandidate sample:')
print(json.dumps(field_value_candidates[:5], indent=2, ensure_ascii=False))